# UNO Vision - main pipeline

Orchestration notebook. All logic lives in `src/`; this notebook only wires modules together.



## A. Setup

In [1]:
import sys
from pathlib import Path
import torch
import pandas as pd
from src.orchestrator import run_submission
from src.validation import evaluate_backend
from src.classical_backend import detect_cards_classical
from src.yolo_backend import detect_cards_yolo
from src.card_labeler import load_cnn
from src.player_zones import load_zone_fracs, center_roi_frac
sys.path.insert(0, str(Path.cwd()))



from config import (
    TEST_IMAGES, SUBMISSIONS_DIR,
    CNN_CHECKPOINT, YOLO_CHECKPOINT,
    SYMBOL_CROPS_DIR, SYMBOL_CROPS_LABELS,
)

PREPARE_DATA = True
TRAIN_CNN = True
TRAIN_YOLO = True
RUN_BACKEND_CLASSICAL = True
RUN_BACKEND_YOLO = True
RUN_VALIDATION = True

device = 'cuda' if torch.cuda.is_available() else 'cpu'
zone_fracs = load_zone_fracs()
center_roi = center_roi_frac(zone_fracs)
print('device:', device)
print('zones:', zone_fracs)
print('center ROI:', center_roi)

device: cpu
zones: {1: (0.28585707146426786, 0.7054845980465815, 0.7006496751624188, 1.0), 2: (0.7546226886556722, 0.17543200601051842, 1.0, 0.7531930879038317), 3: (0.28660669665167415, 0.0, 0.7011494252873564, 0.29564237415477085), 4: (0.0, 0.23441021788129227, 0.2318840579710145, 0.7915101427498121)}
center ROI: (0.2318840579710145, 0.29564237415477085, 0.7546226886556722, 0.7054845980465815)


## B. Symbol crop preparation

In [2]:
if PREPARE_DATA:
    !.venv/bin/python scripts/prepare_symbol_crops.py

Crops written: 15577
         0: 1105
         1: 1218
         2: 1184
         3: 1213
         4: 1088
         5: 1198
         6: 1121
         7: 1099
         8: 1118
         9: 1176
        +2: 1189
        +4: 271
      wild: 295
      skip: 1122
   reverse: 1180


## C. Symbol CNN training

In [ ]:
if TRAIN_CNN:
    !.venv/bin/python hybric_detection_utils/train_uno.py train \
        --image_dir data/symbol_crops \
        --labels data/symbol_crops/labels.csv \
        --output models/uno_symbol_cnn.pt \
        --epochs 30 --batch_size 64

Total usable images: 15577
Class distribution:
         0: 1105
         1: 1218
         2: 1184
         3: 1213
         4: 1088
         5: 1198
         6: 1121
         7: 1099
         8: 1118
         9: 1176
        +2: 1189
        +4: 271
      wild: 295
      skip: 1122
   reverse: 1180

Train: 13240  |  Val: 2337

Model parameters: 4,822,479  (4.82M)

/Users/anthoninduval/Desktop/Bureau - MacBook Air de Anthonin/EPFL/ma2/iapr_uno_competition/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


## D-E. Detection backends

## E. YOLO training (gated)

In [ ]:
if TRAIN_YOLO:
    !.venv/bin/python scripts/train_yolo.py

Traceback (most recent call last):
  File "/Users/anthoninduval/Desktop/Bureau - MacBook Air de Anthonin/EPFL/ma2/iapr_uno_competition/scripts/train_yolo.py", line 13, in <module>
    from ultralytics import YOLO
ModuleNotFoundError: No module named 'ultralytics'


## K. Run submission for both backends

In [ ]:

if RUN_BACKEND_CLASSICAL:
    run_submission(
        image_dir=TEST_IMAGES,
        out_csv=SUBMISSIONS_DIR / 'classical.csv',
        detect_fn=detect_cards_classical,
        cnn=cnn, classes=cnn_classes, device=device,
    )

if RUN_BACKEND_YOLO:
    run_submission(
        image_dir=TEST_IMAGES,
        out_csv=SUBMISSIONS_DIR / 'yolo.csv',
        detect_fn=detect_cards_yolo,
        cnn=cnn, classes=cnn_classes, device=device,
    )

NameError: name 'cnn' is not defined

## Backend agreement check

In [ ]:
if RUN_BACKEND_CLASSICAL and RUN_BACKEND_YOLO:
    a = pd.read_csv(SUBMISSIONS_DIR / 'classical.csv').set_index('image_id')
    b = pd.read_csv(SUBMISSIONS_DIR / 'yolo.csv').set_index('image_id')
    same = (a == b).all(axis=1)
    print(f'Row-level agreement: {same.mean():.3f} ({same.sum()}/{len(same)})')
    for col in a.columns:
        agree = (a[col] == b[col]).mean()
        print(f'  {col}: {agree:.3f}')

## L. Validation on synthetic held-out slice

In [ ]:

if RUN_VALIDATION:
    if RUN_BACKEND_CLASSICAL:
        r = evaluate_backend(detect_fn=detect_cards_classical, cnn=cnn,
                              classes=cnn_classes, device=device)
        print('CLASSICAL:', r)
    if RUN_BACKEND_YOLO:
        r = evaluate_backend(detect_fn=detect_cards_yolo, cnn=cnn,
                              classes=cnn_classes, device=device)
        print('YOLO:', r)